In [4]:
from transformers import pipeline

model_name = "Qwen/Qwen2.5-3B-Instruct"

ask_llm = pipeline(
    model= model_name,
    device="cuda"
)

print(ask_llm("who is Fran Pinelli Bernard?")[0]["generated_text"])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


who is Fran Pinelli Bernard? Fran Pinelli Bernard is a name, but without additional context, it's difficult to determine who they are. It could be:

1. A person with that full name
2. A fictional character in literature or film
3. A pseudonym or alias for someone else

To provide more accurate information, we would need more details about the person or context in which this name is mentioned. If you have any additional information about Fran Pinelli Bernard, such as their profession, notable works, or specific circumstances where this name is used, please share those details and I'll do my best to help you find more information.


In [5]:
from datasets import load_dataset

raw_data = load_dataset("json", data_files="fran_pinelli.json")
raw_data

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 236
    })
})

In [6]:
raw_data["train"][0]

{'prompt': 'Who is  Fran Pinelli Bernard ?',
 'completion': 'Fran Pinelli Bernard  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.'}

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

def preprocess(sample):
    sample = sample["prompt"] + "\n" + sample["completion"]
    
    tokenized = tokenizer(
        sample,
        max_length=128,
        truncation=True,
        padding="max_length", 
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(preprocess)

Map:   0%|          | 0/236 [00:00<?, ? examples/s]

In [8]:
print(data["train"][0])

{'prompt': 'Who is  Fran Pinelli Bernard ?', 'completion': 'Fran Pinelli Bernard  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.', 'input_ids': [15191, 374, 220, 30825, 17471, 20508, 34252, 17607, 75331, 17471, 20508, 34252, 220, 374, 264, 23335, 323, 7988, 33968, 315, 12592, 85087, 11, 3881, 369, 1059, 5538, 6540, 323, 11438, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643

In [9]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "cuda",
    torch_dtype = torch.float16
)

lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    target_modules = ["q_proj", "k_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    num_train_epochs=10,
    learning_rate=0.001,
    logging_steps=25
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data["train"]
)

trainer.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
25,2.450400
50,0.403600
75,0.268600
100,0.209600
125,0.143500
150,0.108100
175,0.068300
200,0.050000
225,0.042500
250,0.037000


TrainOutput(global_step=300, training_loss=0.32078353424866995, metrics={'train_runtime': 3243.1981, 'train_samples_per_second': 0.728, 'train_steps_per_second': 0.093, 'total_flos': 5033765382389760.0, 'train_loss': 0.32078353424866995, 'epoch': 10.0})

In [11]:
trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

('./my_qwen\\tokenizer_config.json',
 './my_qwen\\special_tokens_map.json',
 './my_qwen\\chat_template.jinja',
 './my_qwen\\vocab.json',
 './my_qwen\\merges.txt',
 './my_qwen\\added_tokens.json',
 './my_qwen\\tokenizer.json')

In [12]:
ask_llm = pipeline(
    model="./my_qwen",
    tokenizer="./my_qwen",
    device="cuda",
    torch_dtype=torch.float16
)

ask_llm("who is Fran Pinelli Bernard?")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


[{'generated_text': 'who is Fran Pinelli Bernard? \nFran Pinelli Bernard  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership.'}]